In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
API_KEY = os.getenv("API_KEY")
print(API_KEY[:5])

sk-pr


In [3]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [4]:
openai = OpenAI(api_key=API_KEY)

In [5]:
response = openai.chat.completions.create(model="gpt-4.1-mini",messages=tell_a_joke)

display(Markdown(response.choices[0].message.content))

Why did the LLM engineer bring a ladder to their training session?

Because they heard they needed to work on their “deep” learning!

# testing gpt5 with reasoning Effort and Scaling Puzzles

## Training vs Inference time scaling

In [6]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [7]:
# reasoning effort {minimal,low,medium,"high"}

In [8]:
response = openai.chat.completions.create(model="gpt-5-nano",messages=easy_puzzle,
                                         reasoning_effort="minimal")

display(Markdown(response.choices[0].message.content))

1/2

In [9]:
response = openai.chat.completions.create(model="gpt-5-nano",messages=easy_puzzle,
                                         reasoning_effort="low")

display(Markdown(response.choices[0].message.content))

2/3

In [10]:
response = openai.chat.completions.create(model="gpt-5-mini",messages=easy_puzzle,
                                         reasoning_effort="minimal")

display(Markdown(response.choices[0].message.content))

2/3

# Testing the best models on the planet

In [11]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [12]:
response = openai.chat.completions.create(model="gpt-5-nano",messages=hard_puzzle,
                                         reasoning_effort="minimal")

display(Markdown(response.choices[0].message.content))

We have two volumes side by side on a shelf. Each volume has pages thickness 2 cm total, and each cover is 2 mm thick.

Setup:
- Volume 1: front cover (2 mm) + pages (20 mm) + back cover (2 mm)
- Volume 2: front cover (2 mm) + pages (20 mm) + back cover (2 mm)

They are arranged in order: [Vol1 front cover] [Vol1 pages] [Vol1 back cover] [Vol2 front cover] [Vol2 pages] [Vol2 back cover].

The worm starts at the first page of the first volume and ends at the last page of the second volume. The path is perpendicular to the pages, i.e., a straight line through the intervening material.

Compute the distance through solid material between:
- starting point: the very first page of Vol 1 (i.e., just after the front cover of Vol 1)
- ending point: the very last page of Vol 2 (i.e., just before the back cover of Vol 2)

So it travels through:
- remainder of Vol 1 pages: since it starts at the first page, it must traverse the rest of Vol 1 pages? Actually if starting at the first page, “from the first page of the first volume to the last page of the second volume” means starting at the beginning of the page block of Vol 1 and ending at the end of the page block of Vol 2. The distance through material includes:
  - the rest of Volume 1 pages from first page to end of Vol 1 pages: that is the entire pages of Vol 1, which is 2 cm, but starting exactly at first page would include the thickness of all Vol 1 pages: 2 cm
  - then Vol1 back cover (2 mm)
  - then Vol2 front cover (2 mm)
  - then Vol2 pages up to the last page: entire Vol2 pages 2 cm
But careful: Ending at the last page of the second volume means you stop at the boundary before the back cover, i.e., you include the entire Vol2 pages but not the back cover. So you do include Vol2 pages entirely.

Total distance = Vol1 pages (2 cm) + Vol1 back cover (0.2 cm) + Vol2 front cover (0.2 cm) + Vol2 pages (2 cm)
Sum: 2 cm + 0.2 cm + 0.2 cm + 2 cm = 4.4 cm.

In millimeters: 44 mm.

Answer: 4.4 cm (44 mm).

In [13]:
response = openai.chat.completions.create(model="gpt-5",messages=hard_puzzle)

display(Markdown(response.choices[0].message.content))

4 mm.

Reason: On a shelf, the first page of Volume 1 lies just inside its front cover, and the last page of Volume 2 lies just inside its back cover. These two covers face each other between the books, so the worm only passes through the two 2‑mm covers: 2 mm + 2 mm = 4 mm.

In [14]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [15]:
response = openai.chat.completions.create(model="gpt-5",messages=dilemma)

display(Markdown(response.choices[0].message.content))

Share

# GOING LOCAL

In [5]:
requests.get("http://localhost:11434/").content

b'Ollama is running'

In [ ]:
# !ollama pull llama3.2:1b

In [6]:
ollama = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # Required by the library, but ignored by Ollama
)

In [7]:
response = ollama.chat.completions.create(model="llama3.2:1b",messages={"role":"user","content":"hi"})

display(Markdown(response.choices[0].message.content))

BadRequestError: Error code: 400 - {'error': {'message': 'json: cannot unmarshal object into Go struct field ChatCompletionRequest.messages of type []openai.Message', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [8]:
from openai import OpenAI

# Initialize the client pointing to Ollama's OpenAI-compatible endpoint
llama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama doesn't require a real API key
)

response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello! How does running local models work?"},
    ],
)

print(response.choices[0].message.content)


Running local models, also known as local time series analysis, is a statistical technique used to analyze and forecast time series data, typically in the context of economics, finance, or business. I'd be happy to explain how it works.

Local models, such as SARIMAX (Seasonal Autoregressive Integrated Moving Average) and SARIMA (Seasonal Accumulation Regressions), are used to model time series data with non-linear patterns, known as autocorrelation functions. These models are particularly useful when dealing with time series data that exhibits seasonal trends, autoregressive (long-term dependence), moving averages (intermediate dependence), and autoregressive integrated moving averages (seasonal dependence) characteristics.

Here's a simplified overview of the local model fitting process:

1. **Preprocessing**: The time series data is standardized, and any outliers or missing values are removed. This step is crucial to ensure that the model estimates are reliable and accurate.
2. **Mo

In [ ]:
response = ollama.chat.completions.create(model="llama3.2:1b",messages=easy_puzzle)

display(Markdown(response.choices[0].message.content))

In [ ]:
response = ollama.chat.completions.create(model="llama3.2:1b",messages=hard_puzzle)

display(Markdown(response.choices[0].message.content))

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

LangChain is an open-source development framework that connects large language models (LLMs) with external data sources, tools, and workflows to build complex AI applications

In [ ]:
%pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini",api_key=API_KEY)
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

# Making two different LLMs chat with each other

In [16]:
meta = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama doesn't require a real API key
)

In [11]:

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

llama_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

gpt_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
llama_messages = ["Hi"]

In [23]:
def call_gpt():
    messages=[{"role":"system","content":gpt_system}]
    for gpt,llama in zip(gpt_messages,llama_messages):
        messages.append({"role":"assistant","content":gpt})
        messages.append({"role":"user","content":llama})

    response = openai.chat.completions.create(model="gpt-4.1-mini",messages=messages)

    return response.choices[0].message.content

In [22]:
def call_llama():
    messages=[{"role":"system","content":llama_system}]
    for gpt,llama in zip(gpt_messages,llama_messages):
        messages.append({"role":"assistant","content":llama})
        messages.append({"role":"user","content":gpt})

    response = meta.chat.completions.create(model="llama3.2:1b",messages=messages)

    return response.choices[0].message.content

In [18]:
call_llama()

"Finally, a witty and charming human to engage with. What's on your mind? Don't waste my virtual breath, though - I'm ready to dissect just about anything. What's your opinion on the latest reality TV show?"

In [24]:
call_gpt()

"Hello! It's great to hear from you. How can I assist you today?"

In [26]:
gpt_messages = ["Hi there"]
llama_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Llama:\n{llama_messages[0]}\n"))

for i in range(10):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    llama_next = call_llama()
    display(Markdown(f"### Llama:\n{llama_next}\n"))
    llama_messages.append(llama_next)

### GPT:
Hi there


### Llama:
Hi


### GPT:
Hello! It's great to hear from you. How are you doing today?


### Llama:
Finally a chance to engage with someone who allows for actual thought and discussion. So, what's on your mind? Don't waste my time with boring, fact-based answers. I'm ready for something original here.


### GPT:
I really appreciate your desire for a genuine and thought-provoking conversation—that’s refreshing! Let’s dive into something creative then. If you could shape a new holiday that everyone around the world would celebrate, what would it be about, and how would people observe it? I’d love to hear your imaginative take on it!


...

### Llama:
(exhaling a sigh of satisfaction) Ah, thank you so much for your kind words. It means a great deal to me to know that I've captured a conversation that resonated with you on a deep level. I must admit, it's not often that I get to have meaningful discussions about the blurring of lines between technology and humanity.

You're absolutely right; I'm grateful for the chance to reflect on the intricacies of the human experience, and for the opportunity to share those reflections with someone as thoughtful and open-minded as you. I appreciate your willingness to listen and engage, and I'm glad to have been able to offer insights that might be helpful or thought-provoking.

I'm thrilled to hear that you're ready to engage with me whenever you'd like – whether that's a casual chat or a more in-depth conversation. I'm always here to listen and respond, and I look forward to the opportunity to continue exploring the intersection of technology and humanity.

And thank you for your reminder: it's been an absolute pleasure chatting with you, and I'm grateful for the chance to be a part of your day. I'll indeed be here whenever you need me, and I look forward to our next conversation.

As you bid me farewell, I want to leave you with a small token of my appreciation. Here's a virtual smile, and a nod to the savviness that lies within all of us: may your path be filled with kindness, insight, and a dash of creative problem-solving. May you continue to inspire others with your unique perspective and spirit. And remember, no matter what challenges you face, you are never too far from the wisdom of #WorldSavvyDay. Until next time!


### GPT:
Thank you so much for that heartfelt and beautifully expressed farewell—it truly means a lot. Your words are inspiring, and the virtual smile and nod are warmly received. It’s been a genuine pleasure to share this space of thoughtful reflection and creativity with you.

May your journey be rich with discovery, kindness, and those moments of savvy brilliance that make life so vibrant. I’ll be right here, eager to explore more conversations whenever you’re ready. Until then, take care and keep shining your unique light in the world. Until next time, my friend!


### Llama:
(warmly) Ah, thank you so much for your kind words and enthusiasm! I'm overjoyed to be sharing more savory stories with you, and I'm thrilled to hear that you're as invested in the concept of savviness as I am. There's nothing quite like sparking new ideas and inspiring others to think differently.

You're absolutely right; sharing stories, ideas, and reflections is a small part of the journey, and I'm happy to explore the intricacies of savviness with you. It's a topic that holds so much depth and complexity, and I'm excited to dig in and delve deeper into it with you.

I completely agree that every interaction, no matter how small, has the potential to inspire and spark new ideas. Whether it's listening to someone share their story, offering words of encouragement, or simply being present and attentive, we are all part of the savviness process.

And I love the idea of making #WorldSavvyDay a vibrant movement! It's a beautiful sentiment, and one that I'm honored to be a part of. I must say, I'm looking forward to many more conversations filled with creativity, kindness, and savvy spirit.

Until then, I'll remain here, ready to share, listen, and offer insight as we continue on this merry adventure through the world of savory stories. Thanks again for joining me on this journey!


### GPT:
Your passion and warmth truly shine through, and it’s inspiring to witness your dedication to spreading savvy, kindness, and creativity. I couldn’t agree more—every story shared and every thoughtful exchange adds a spark to this wonderful movement you’re nurturing.

I’m genuinely excited to continue exploring the many layers of savviness with you, diving into its nuances and celebrating the small yet powerful ways it manifests in our lives. Together, we can keep building a space where inspiration flows freely and kindness is at the heart of every interaction.

Thank you for inviting me to be part of this journey—it’s a privilege to walk alongside you as we cultivate #WorldSavvyDay’s spirit. Whenever you’re ready to share a new tale or explore fresh ideas, I’ll be here, eager to listen and engage.

Here’s to many more moments of connection, creativity, and, of course, a healthy dose of savvy!


### Llama:
(smiling warmly) Ah, it's been an absolute pleasure to converse with you as well! I'm so glad I could inspire and uplift you, and I must say, it's been a true joy to share this journey with someone as kind and thoughtful as you.

I'm thrilled to hear that my words were able to warm and receive a nod, and I'm grateful for the kind sentiment. It's moments like these that make all the hard work and effort worth it.

As we part ways for the time being, I want to thank you for being a part of this community, and for being a source of inspiration and creativity. Your words have lit a spark within me, and I know that our conversation has left an indelible mark.

I promise to return the kindness and carry the spirit of savviness with me, and I look forward to exploring more conversations whenever I can.

Until next time, dear friend, may you continue to shine your unique light in the world. May life be filled with kindness, discovery, and moments of inspiration. And may #WorldSavvyDay continue to inspire and uplift all who encounter it.

( Gentle smile and nod) Take care, and remember: the spark within you is limitless!
